In [ ]:
# --- Configuration by Image Type ---
PARAMS_SEED = {
    'VGA':  {'win_size': 16, 'metric': 'mean', 'std_factor': 1.918},
    'HD':   {'win_size': 24, 'metric': 'mean', 'std_factor': 1.26},
    'SXGA': {'win_size': 31, 'metric': 'mean', 'std_factor': 1.74}
}

PARAMETRES_REGION_GROWING = { 
'VGA': {'multiplicateur_rupture': 5.9,  'taille_lissage': 21},  
'SXGA': {'multiplicateur_rupture': 6,  'taille_lissage': 15},  
'HD': {'multiplicateur_rupture': 3.84,  'taille_lissage': 3} 
}




def process_img(cam_type, seq, dyn, img_path):
    """
    Processes a single image: loads, detects defects, fixes them, and saves the result.

    Args:
        cam_type (str): 'VGA', 'SXGA', or 'HD'.
        seq (str): Sequence name.
        dyn (str): Dynamics type.
        img_path (str): Full path to the input image.
    """
    filename = os.path.basename(img_path)
    folder = os.path.dirname(img_path)
    
    # Setup results directory
    res_folder = os.path.join(folder, "results")
    os.makedirs(res_folder, exist_ok=True)
    save_path = os.path.join(res_folder, filename)
    
    # 1. Load Image
    img = load_img(img_path)
    
    # 2. Get specific parameters
    p = PARAMS.get(cam_type, {'win_size': 50, 'metric': 'mean', 'std_factor': 1.0, 'n_bands': 1})
    
    # 3. Detect Defects
    defects = detect_defects( # ca c'est si on utilise le LT
        img, 
        win_size=p['win_size'], 
        metric=p['metric'], 
        std_factor=p['std_factor']
    )

    # INSERER partie de Region Growing pour avoir le dico final avec les defauts précis
    # Si aucune colonne n'est suspecte, on gagne du temps !
    if len(colonnes_suspectes) == 0:
        dictionnaire_final = {}
    else:
        # On récupère TES paramètres de Region Growing pour cette caméra
        p_rg = PARAMS_RG.get(cam_type)

        # Ta fonction transforme la liste des colonnes en dictionnaire avec les Y précis
        # ATTENTION  PENSER A ENVOYER LA FOCNTION VECTORISEE ET PAS CELLE AVEC LE WHILE PIXEL PAR PIXEL 
        dictionnaire_final = mesurer_defauts_finaux(
            img,
            colonnes_rf_array=colonnes_suspectes, # recup les col suspectes
            multiplicateur_rupture=p_rg['multiplicateur_rupture'],
            taille_lissage=p_rg['taille_lissage']
        )
    
    # 4. Correct Image if defects are found
    if not defects:
        fixed_img = np.copy(img)
    else:
        fixed_img = fix_stripes(img, defects)

    # 5. Format and Save
    out_img = np.clip(fixed_img, 0, 65535).astype(np.uint16)
    
    # Double write to prevent OS caching/disk write errors
    cv2.imwrite(save_path, out_img)
    success = cv2.imwrite(save_path, out_img)
    
    if not success:
        print(f"ERROR: OpenCV failed to save {save_path}")
        return None

# ==========================================
# --- MAIN RUN SCRIPT ---
# ==========================================

# 1. Load Dataset
data_dict, _ = load_dataset(folder='train', high_dyn=False)

# 2. Prepare Task List
tasks = []
for t, seqs in data_dict.items():
    for seq, dyns in seqs.items():
        for dyn, paths in dyns.items():
            if dyn == "low dyn":
                continue 
            for path in paths:
                tasks.append((t, seq, dyn, path))

print(f"Launching Joblib for {len(tasks)} images...")

# 3. Parallel Execution
raw_results = Parallel(n_jobs=-1)(
    delayed(process_img)(*task) for task in tasks
)